In [ ]:
import torch
import sys
sys.path.insert(0, "/root/vk_work")
import torch.nn.functional as F
from torch import nn

from src.model import CustomResNet

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

class Clf(nn.Module):
    def __init__(self, model, simkin=True):
        super().__init__()
        self.model, self.simkin = model, simkin
        self.register_buffer("mean", MEAN)
        self.register_buffer("std", STD)
    def forward(self, x):
        x = (x - self.mean) / self.std
        return self.model(x, mode='clas') if self.simkin else self.model(x)

In [2]:
from torch.utils.data import DataLoader
from torchvision import transforms
from datasets import load_dataset
from src.custom_datasets import STL10RGBDataset

t = transforms.ToTensor()
test = STL10RGBDataset(load_dataset("jxie/stl10")["test"], transform=t)
test_loader = DataLoader(test, batch_size=128, shuffle=False, num_workers=8)

/root/vk_work/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def load(ckpt):
    m = CustomResNet().to(device)
    m.load_state_dict(torch.load(ckpt, map_location=device))
    return Clf(m, simkin=True).to(device).eval()

reg = load("../models/ResNetSIM_best.pth")
base = load("../models/ResNet_best.pth")

/root/vk_work/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/root/vk_work/.venv/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [4]:
import torchattacks
from torch.utils.data import Subset, DataLoader

sub = Subset(test, range(1000))
sub_loader = DataLoader(sub, batch_size=128, shuffle=False, num_workers=8)

def adv_acc(clf, loader, atk=None):
    correct = total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        if atk is not None:
            x = atk(x, y)
        with torch.no_grad():
            correct += (clf(x).argmax(1) == y).sum().item()
        total += y.size(0)
    return correct / total

for name, clf in [("baseline", base), ("simkin", reg)]:
    print(f"\n{name}: clean {adv_acc(clf, sub_loader):.3f}")
    for eps in [1/255, 2/255, 4/255, 8/255]:
        f = adv_acc(clf, sub_loader, torchattacks.FGSM(clf, eps=eps))
        p = adv_acc(clf, sub_loader, torchattacks.PGD(
            clf, eps=eps, alpha=eps/4, steps=10, random_start=True))
        print(f"  eps={eps*255:.0f}/255  FGSM {f:.3f}  PGD {p:.3f}")


baseline: clean 0.944
  eps=1/255  FGSM 0.669  PGD 0.546
  eps=2/255  FGSM 0.451  PGD 0.149
  eps=4/255  FGSM 0.236  PGD 0.000
  eps=8/255  FGSM 0.089  PGD 0.000

simkin: clean 0.938
  eps=1/255  FGSM 0.680  PGD 0.555
  eps=2/255  FGSM 0.442  PGD 0.149
  eps=4/255  FGSM 0.213  PGD 0.001
  eps=8/255  FGSM 0.087  PGD 0.000
